In [1]:
import numpy as np


# 1. 定义基础参数
d = 64      # 向量维度（比如ViT-Base输出768维，这里简化为64维）
nb = 100000  # 向量库的总数据量（模拟10万条候选向量）
nq = 10000   # 待检索的查询向量数量（模拟1万次查询）

# 2. 设置随机种子，保证结果可复现
np.random.seed(1234)

# 3. 生成向量库（xb）和查询向量（xq）
# 注意：Faiss要求所有向量必须是float32类型，这一点非常重要！
xb = np.random.random((nb, d)).astype('float32')
xq = np.random.random((nq, d)).astype('float32')

# 4. 给向量加一点“人工区分度”（仅用于测试，实际工程中不需要）
# 让向量的第一维随索引递增，这样每个向量都有明显的差异，方便验证检索结果是否正确
xb[:, 0] += np.arange(nb) / 1000.
xq[:, 0] += np.arange(nq) / 1000.


In [2]:
import faiss


# 1. 构建索引：IndexFlatL2
# - IndexFlat：表示这是“Flat索引”（暴力检索，无优化）
# - L2：表示相似度度量方法为L2范数（即欧氏距离，距离越小越相似）
index = faiss.IndexFlatL2(d)

# 2. 检查索引是否需要训练
# Flat索引不需要训练，直接添加向量即可，所以输出为True
print(f"索引是否需要训练: {index.is_trained}")

# 3. 将向量库添加到索引中
index.add(xb)

# 4. 查看索引中已添加的向量总数
print(f"索引中已添加的向量总数: {index.ntotal}")  # 输出为100000，和我们定义的nb一致

索引是否需要训练: True
索引中已添加的向量总数: 100000


In [3]:
k = 4  # TopK的K值，即每个查询向量找最相似的4个结果

# 执行检索
# - 输入：xq（待检索向量）、k（TopK数量）
# - 输出：
#   D：距离矩阵，形状为(nq, k)，每一行对应一个查询向量的TopK距离（从小到大排列）
#   I：索引矩阵，形状为(nq, k)，每一行对应一个查询向量的TopK向量在原库中的索引
D, I = index.search(xq, k)

# 查看前5个查询向量的TopK索引
print(f"前5个查询向量的TopK索引: \n{I[:5]}")
# 查看后5个查询向量的TopK距离
print(f"前5个查询向量的TopK距离: \n{D[:5]}")


前5个查询向量的TopK索引: 
[[ 381  207  210  477]
 [ 526  911  142   72]
 [ 838  527 1290  425]
 [ 196  184  164  359]
 [ 526  377  120  425]]
前5个查询向量的TopK距离: 
[[6.815506  6.8894653 7.3956795 7.4290257]
 [6.6041145 6.679695  6.7209625 6.828682 ]
 [6.4703865 6.8578568 7.0043793 7.036564 ]
 [5.573681  6.4075394 7.1395187 7.3555984]
 [5.409401  6.2322083 6.4173393 6.5743713]]


In [4]:
# 手动计算 xq[0] 和 xb[381] 的 L2 距离
manual_distance = np.linalg.norm(xq[0] - xb[381])
print(f"手动计算的L2距离: {manual_distance}")
print(f"Faiss返回的Top1距离: {D[0][0]}")

手动计算的L2距离: 2.6106534004211426
Faiss返回的Top1距离: 6.8155059814453125


In [5]:
# 【方法1】取平方根
print(f"Faiss结果的平方根: {D[0][0] ** 0.5}")
print(f"平方根后的结果是否相近: {np.isclose(D[0][0] ** 0.5, manual_distance)}")

# 【方法2】取平方
print(f"手动计算结果的平方: {manual_distance ** 2}")
print(f"平方后的结果是否相近: {np.isclose(D[0][0], manual_distance ** 2)}")

Faiss结果的平方根: 2.6106524053280844
平方根后的结果是否相近: True
手动计算结果的平方: 6.815511177130475
平方后的结果是否相近: True


In [6]:
import faiss


# 三个核心参数
dim     = 64               # 向量维度
param   = 'Flat'           # 索引类型字符串（核心参数）
measure = faiss.METRIC_L2  # 距离度量方式（常用L2或内积）

# 构建索引
index = faiss.index_factory(dim, param, measure)

In [7]:
dim                 = 64               # 向量维度
construction_method = 'Flat'           # 索引类型字符串（核心参数）
measure             = faiss.METRIC_L2  # 距离度量方式（常用L2或内积）

# 构建索引
index = faiss.index_factory(dim, construction_method, measure)

print(f"构建的索引是否需要训练：{index.is_trained}")  # 输出True，不需要训练
print(f"当前索引中向量总数：{index.ntotal}")  # 输出向量总数

# 随机创建一个向量
import numpy as np
xb = np.random.random((1024, dim)).astype("float32")

# 直接添加新向量
index.add(xb)  # 直接添加向量
print(f"添加新向量后索引中向量总数：{index.ntotal}")  # 输出向量总数

构建的索引是否需要训练：True
当前索引中向量总数：0
添加新向量后索引中向量总数：1024


In [8]:
import numpy as np


dim                 = 64               # 向量维度
construction_method = "IVF100,Flat"    # 100个聚类中心，桶内用Flat精确检索
measure             = faiss.METRIC_L2

# 构建索引
index = faiss.index_factory(dim, construction_method, measure)
print(f"构建的索引是否需要训练：{index.is_trained}")  # 输出False，需要先训练

# 随机创建向量
xb = np.random.random((102400, dim)).astype("float32")

# 用向量库训练k-means
index.train(xb)
index.add(xb)  # 训练后再添加向量

# 调优nprobe（可选，根据需求调整）
index.nprobe = 10

构建的索引是否需要训练：False


In [ ]:
import numpy as np


dim                 = 64               # 向量维度
construction_method = "PQ16"           # 把向量切成16段（64/16=4，每段4维）
measure             = faiss.METRIC_L2

# 构建索引
index = faiss.index_factory(dim, construction_method, measure)
print(f"构建的索引是否需要训练：{index.is_trained}")  # 输出False，需要先训练

# 随机创建向量
xb = np.random.random((102400, dim)).astype("float32")

# 用向量库训练PQ
index.train(xb)
index.add(xb)  # 训练后再添加向量

构建的索引是否需要训练：False


In [10]:
import numpy as np


dim                 = 64               # 向量维度
construction_method = "IVF100,PQ16"    # 把向量分为100个聚类中心，每个聚类中心用16段PQ编码
measure             = faiss.METRIC_L2

# 构建索引
index = faiss.index_factory(dim, construction_method, measure)
print(f"构建的索引不需要训练：{index.is_trained}")  # 输出False，需要先训练

# 随机创建向量
xb = np.random.random((102400, dim)).astype("float32")

# 用向量库训练PQ
index.train(xb)
index.add(xb)  # 训练后再添加向量

index.nprobe = 10  # 调优nprobe（可选，根据需求调整）

构建的索引不需要训练：False


In [12]:
import numpy as np


dim                 = 64               # 向量维度
construction_method = "LSH"
measure             = faiss.METRIC_L2

# 构建索引
index = faiss.index_factory(dim, construction_method, measure)
print(f"构建的索引不需要训练：{index.is_trained}")  # 输出False -> 不需要训练

# 随机创建向量
xb = np.random.random((102400, dim)).astype("float32")

# 直接添加向量即可
index.add(xb)

构建的索引不需要训练：True


In [ ]:
import numpy as np


dim                 = 64               # 向量维度
construction_method = "HNSW64"         # 每个节点最多连64条边
measure             = faiss.METRIC_L2

# 构建索引
index = faiss.index_factory(dim, construction_method, measure)
print(f"构建的索引不需要训练：{index.is_trained}")

# 随机创建向量
xb = np.random.random((102400, dim)).astype("float32")

# HNSWx的特点是不需要训练，训练是在add时进行
index.add(xb)

构建的索引不需要训练：True
